In [92]:
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [93]:
X_train, X_test, y_train, y_test = joblib.load('data/processed_data.pkl')
preprocessor = joblib.load('models/preprocessor.pkl')

In [94]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

In [95]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_model = None
best_cv_accuracy = 0
best_model_name = ""

In [96]:
for name, model in models.items():
    print(f"--- {name} ---")
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    cv_mean = np.mean(cv_scores)
    cv_std = np.std(cv_scores)

    print(f"CV Accuracy (mean): {cv_mean:.4f} (+/- {cv_std:.4f})")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Test Accuracy:         {accuracy:.4f}")
    print(f"Test Precision:        {precision:.4f}")
    print(f"Test Recall:           {recall:.4f}")
    print(f"Test F1-Score:         {f1:.4f}\n")

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("="*50 + "\n")

    if cv_mean > best_cv_accuracy:
        best_cv_accuracy = cv_mean
        best_model = model
        best_model_name = name

print(f"Best model: {best_model_name}")

--- Logistic Regression ---
CV Accuracy (mean): 0.8745 (+/- 0.0015)
Test Accuracy:         0.8762
Test Precision:        0.8720
Test Recall:           0.8395
Test F1-Score:         0.8554

Confusion Matrix:
[[10595  1118]
 [ 1455  7613]]

--- Random Forest ---
CV Accuracy (mean): 0.9625 (+/- 0.0010)
Test Accuracy:         0.9634
Test Precision:        0.9741
Test Recall:           0.9412
Test F1-Score:         0.9574

Confusion Matrix:
[[11486   227]
 [  533  8535]]

--- Gradient Boosting ---
CV Accuracy (mean): 0.9413 (+/- 0.0015)
Test Accuracy:         0.9413
Test Precision:        0.9465
Test Recall:           0.9173
Test F1-Score:         0.9317

Confusion Matrix:
[[11243   470]
 [  750  8318]]

--- K-Nearest Neighbors ---
CV Accuracy (mean): 0.9268 (+/- 0.0024)
Test Accuracy:         0.9301
Test Precision:        0.9492
Test Recall:           0.8874
Test F1-Score:         0.9172

Confusion Matrix:
[[11282   431]
 [ 1021  8047]]

Best model: Random Forest


In [97]:
grid_space = {'n_estimators': [50, 100, 200], 'max_depth': [None, 3, 5, 8]}
grid = GridSearchCV(estimator=best_model, param_grid=grid_space, cv=cv, scoring='accuracy', n_jobs=-1)
model_grid = grid.fit(X_train, y_train)

In [98]:
print(f"Grid search: {grid.best_params_}, score: {grid.best_score_}")

Grid search: {'max_depth': None, 'n_estimators': 200}, score: 0.9627540137639576


In [99]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',grid.best_estimator_)
])

In [100]:
joblib.dump(final_pipeline, 'models/final_pipeline.pkl')

['models/final_pipeline.pkl']